# analysis.organism-range

In this notebook we will study the *organism range*, a property that could be seen as the analogue of organism *host-range* in hosts. 

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import numpy as np
import matplotlib.pyplot as plt
from daforfer import DaforferDB
import powerlaw
import networkx as nx
plt.rcParams['svg.fonttype'] = 'none'
from yaml import load, Loader
from daforfer import DaforferDB
conf = load(open("conf.yaml"), Loader)
db = DaforferDB(conf['database'])
si = DaforferDB(conf['si'])
si.toc()

## Load data

In [ ]:
metadata = db.conn.sql('SELECT * FROM D_sites').df()
bacteria_hits = db.conn.sql('SELECT * FROM D_PABHits').df()
virus_hits = db.conn.sql('SELECT * FROM D_virusHits').df()

# bacteria_hits = pd.read_csv("output/hits.bacteria.csv", sep=";").query("is_pab==True")
bacteria_hits = pd.merge(metadata, bacteria_hits, on='library', how='left')#.dropna(subset='taxid')
# bacteria_hits['taxid'] = bacteria_hits['taxid'].astype(int)

# virus_hits = pd.read_csv("output/hits.virus.csv", sep=";")
virus_hits = pd.merge(metadata, virus_hits, on='library', how='left')#.dropna(subset='taxid')
# virus_hits['taxid'] = virus_hits['taxid'].astype(int)

## Organism range calculation

In [ ]:

host_bacteria_range = bacteria_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().value_counts(
        ['host_taxon']
    ).reset_index().rename(columns={'count': 'bacteria_range'})

host_bacteria_range

In [ ]:
host_virus_range = virus_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().value_counts(
        ['host_taxon']
    ).reset_index().rename(columns={'count': 'virus_range'})
host_virus_range

In [ ]:
organism_range = pd.merge(host_bacteria_range, host_virus_range, on='host_taxon', how='outer').fillna(0)
# WE will save this dataframe later
organism_range

Below, we will visualize the organisms range using a histogram.

In [ ]:
organism_range_melt = organism_range.melt(id_vars=['host_taxon'], value_vars=['bacteria_range', 'virus_range'])
organism_range_melt['variable'] = organism_range_melt['variable'].apply(lambda x: x.replace("_range", ""))
organism_range_melt = organism_range_melt.rename(columns={'variable':'kingdom', 'value':'range'})
organism_range_melt

In [ ]:

max_range = (organism_range_melt['range'].max() // 5) + 2
bins = np.arange(0, max_range * 5, 5)
g = sns.displot(data=organism_range_melt, x='range', col='kingdom', height=2.0, bins=bins)
g.axes[0, 0].set_yscale('log')
g.axes[0, 0].set_xticks([0, 30, 60])
g.set_xlabels("Organism range")
g.set_ylabels("Count")
g.savefig("figures/displot.organism-range.colbykingdom.svg")

## Organism range distribution

In [ ]:
heavy_tail_model_lrtest = []

pwl_virus = powerlaw.Fit(data=organism_range_melt.query('kingdom == "virus"').query('range > 0')['range'].values, discrete=True)
pwl_bacteria = powerlaw.Fit(data=organism_range_melt.query('kingdom == "bacteria"').query('range > 0')['range'].values, discrete=True)

R, p = pwl_virus.distribution_compare('power_law', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'virus', 'dist1': 'power-law', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_virus.distribution_compare('lognormal', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'virus', 'dist1': 'lognormal', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_bacteria.distribution_compare('power_law', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'bacteria', 'dist1': 'power-law', 'dist2': 'exponential', 'R':R, 'p-value':p})
R, p = pwl_bacteria.distribution_compare('lognormal', 'exponential')
heavy_tail_model_lrtest.append({'distribution':'bacteria', 'dist1': 'lognormal', 'dist2': 'exponential', 'R':R, 'p-value':p})
heavy_tail_model_lrtest = pd.DataFrame.from_records(heavy_tail_model_lrtest)

db.save_dataframe(
    heavy_tail_model_lrtest, table_name="T_orgRangeslogRatioTest", description="Power-law, Lognormal, and Exponential fit log-ratio test"
)

heavy_tail_model_lrtest

In [ ]:
fig, ax = plt.subplots(1, 2)
fig.set_size_inches(6, 3)
pwl_bacteria.plot_ccdf(ax=ax[0], marker='o')
pwl_bacteria.power_law.plot_ccdf(ax=ax[0], color='black', linestyle='-')
pwl_bacteria.exponential.plot_ccdf(ax=ax[0], color='red', linestyle='-')
pwl_bacteria.lognormal.plot_ccdf(ax=ax[0], color='black', linestyle='--')

pwl_virus.plot_ccdf(ax=ax[1], marker='o')
pwl_virus.power_law.plot_ccdf(ax=ax[1], color='black', linestyle='-')
pwl_virus.exponential.plot_ccdf(ax=ax[1], color='red', linestyle='-')
pwl_virus.lognormal.plot_ccdf(ax=ax[1], color='black', linestyle='--')

# ax[0].set_title("Bacteria range")
# ax[1].set_title("Virus range ")

ax[0].set_xlabel("Bacteria range")
ax[1].set_xlabel("Virus range")

ax[0].set_ylabel("CCDF")
ax[1].set_ylabel("CCDF")

fig.tight_layout()
# fig.savefig("figures/ccdfplot.host-range-distributions.svg")

## Regression analysis between bacteria and virus range

In [ ]:
g = sns.lmplot(data=organism_range, x='bacteria_range', y='virus_range', height=2.0,  scatter_kws={'color':'black', 's':15}, line_kws={'color':'red', 'linestyle':'--'}, truncate=False)
g.set_xlabels("Bacteria range")
g.set_ylabels("Virus range")
g.ax.set_xlim(0, 60)
g.ax.set_ylim(0, 60)
g.ax.set_xticks([0, 20, 40, 60])
g.ax.set_yticks([0, 20, 40, 60])
g.ax.axvline(10, ymin=0.17, ymax=1.0, color='gray', linestyle='--')
g.ax.axhline(10, xmin=0.17, xmax=1.0, color='gray', linestyle='--')
g.savefig("figures/linreg.bact-range.virus-range.svg")

In [ ]:
test_1 = stats.linregress(organism_range['bacteria_range'], organism_range['virus_range'])

test_1_results = pd.DataFrame.from_records([
    {"key": "title", "value":"Regression between bacteria and virus range"},
    {"key": "test-type", "value":"Regression"},
    {"key": "H0", "value":"No correlation between number of libraries and species richness"},
    {"key": "H1", "value":"Correlation between number of libraries and species richness"},
    {"key": "p-value", "value": test_1.pvalue}, # type: ignore
    {"key": "significative", "value": test_1.pvalue < 0.05}, # type: ignore
    {"key": "intercept", "value": test_1.intercept}, # type: ignore
    {"key": "slope", "value": test_1.slope}, # type: ignore
    {"key": "r-value", "value": test_1.rvalue}, # type: ignore
    {"key": "R2", "value": test_1.rvalue ** 2} # type: ignore

])

db.save_dataframe(test_1_results, "T_orgRangeCorr", "Regression between bacteria and virus range")
test_1_results



## Computing total organism range

In [ ]:
organism_range['total'] = organism_range['bacteria_range'] + organism_range['virus_range']
organism_range.sort_values(by='total', ascending=False)

## Cooccurrences at host level

Now, let's also include whether these organisms also include a higher number of cooccurrences. We will load first the cooccurrence network

In [ ]:
G = nx.read_graphml("output/network.coocurrence.virusbact-bylibrary.trans.graphml")
#nx.draw(G)
G.nodes(data=True)

Now, we need to count the number of cooccurrences in each host. For that, first we need to get whole lists of all the organisms detected in a given host. For that, we will convert all organisms detected in a given host into a list. 

In [ ]:

host_bacteria_range_scientific_name = bacteria_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().groupby(
        ['host_taxon'], as_index=False
    )['scientific_name'].apply(list)

host_virus_range_scientific_name = virus_hits.value_counts(
    ['host_taxon', 'scientific_name']
    ).reset_index().groupby(
        ['host_taxon'], as_index=False
    )['scientific_name'].apply(list)

host_organism_range_scientific_name = pd.merge(host_bacteria_range_scientific_name, host_virus_range_scientific_name, on='host_taxon')
host_organism_range_scientific_name

Now, we can make subnetworks out of the original network, and simply count the number of edges.

In [ ]:
host_organism_range_scientific_name['network'] = host_organism_range_scientific_name.apply(
    lambda x: G.subgraph(x.scientific_name_x + x.scientific_name_y), axis=1
)
host_organism_range_scientific_name['n_cooccurrences'] = host_organism_range_scientific_name['network'].apply(lambda x: x.number_of_edges())
host_organism_range_scientific_name

In [ ]:
organism_range = pd.merge(organism_range, host_organism_range_scientific_name[['host_taxon', 'n_cooccurrences']], on='host_taxon', how='outer').fillna(0)
organism_range['all_possible_cooccurrences'] = organism_range['bacteria_range'] * organism_range['virus_range']
organism_range = pd.merge(organism_range, metadata[['host_taxon', 'habitat']].drop_duplicates().groupby('host_taxon', as_index=False)['habitat'].apply(lambda x: ", ".join(list(x))), on='host_taxon')
db.save_dataframe(
    organism_range, table_name="D_organismRange", 
    description="Hosts bacteria and virus ranges"
)
si.save_dataframe(
    organism_range, table_name="Table10", 
    description="Hosts bacteria and virus ranges"
)
organism_range

In [ ]:
g = sns.lmplot(data=organism_range, x='all_possible_cooccurrences', y='n_cooccurrences', height=2.0,  scatter_kws={'color':'black', 's':15}, line_kws={'color':'red', 'linestyle':'--'})
g.set_xlabels("All possible virus-bacteria cooc.")
g.set_ylabels("Detected virus-bacteria cooc.")
g.savefig("figures/linreg.poscooc_cooc.svg")

In [ ]:
test_2 = stats.linregress(organism_range['all_possible_cooccurrences'], organism_range['n_cooccurrences'])

test_2_results = pd.DataFrame.from_records([
    {"key": "title", "value":"Possible cooccurrences versus detected cooccurrences"},
    {"key": "test-type", "value":"Regression"},
    {"key": "H0", "value":"No correlation between number of libraries and species richness"},
    {"key": "H1", "value":"Correlation between number of libraries and species richness"},
    {"key": "p-value", "value": test_2.pvalue}, #type: ignore
    {"key": "significative", "value": test_2.pvalue < 0.05}, #type: ignore
    {"key": "intercept", "value": test_2.intercept}, #type: ignore
    {"key": "slope", "value": test_2.slope}, #type: ignore
    {"key": "r-value", "value": test_2.rvalue}, #type: ignore
    {"key": "R2", "value": test_2.rvalue ** 2} #type: ignore

])

db.save_dataframe(
    test_2_results, table_name="T_hostCooc", 
    description="Correlation test between possible cooccurences and detected cooccurences at host level"
)
test_2_results


In [ ]:
db.conn.close()
si.conn.close()